# Песочница паттернов GoF: Strategy, Factory Method, Observer

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java. Каждый пример представлен **одним файлом** `.java` и запускается без отдельной компиляции командой `java Файл.java` (запуск исходного файла, Java 11 и выше).

**Идея задания.** Паттерн изучается не по описанию, а по разнице: сначала запускается код «до» — работающий, но жёстко связанный; затем тот же сценарий переписывается на паттерн, и проверки доказывают, что **поведение не изменилось**, а **новое требование добавляется без правки старого кода**.

**Состав работы**

| Раздел | Паттерн | Что делает студент |
|---|---|---|
| 2 | Strategy | **образец** — разобран полностью, нужно только запустить и разобраться |
| 3 | Factory Method | задача: самостоятельно переписать код «до» |
| 4 | Observer | задача: самостоятельно переписать код «до» |
| 5–6 | — | сравнение, выводы, контрольные вопросы |

**Правила оформления, от которых зависит автопроверка**

1. Имена файлов строго: `TariffAfter.java`, `FactoryAfter.java`, `ObserverAfter.java`.
2. В каждом файле **первым** идёт открытый класс с методом `main` — иначе запуск исходного файла не работает.
3. Каждый файл сам себя проверяет: печатает строки `[OK]` / `[FAIL]` и бросает `AssertionError`, если хотя бы одна проверка не пройдена.
4. Идентификаторы — латиницей и описательно (`tariffStrategy`, а не `ts`), комментарии — по-русски.
5. К каждому паттерну — диаграмма классов в файле `*.puml`.

**Сдача.** Все файлы `.java`, `.puml`, сам блокнот и вывод ячейки самопроверки — в pull request.

## 1. Подготовка среды

In [1]:
# 1.1 Проверить, какая Java доступна в среде.
# Нужна версия 11 или выше: только с неё работает запуск одного исходного файла (java Файл.java).
!java -version
!javac -version

[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.016s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
javac 21.0.12.1


Если версия ниже 11, запуск одного исходного файла недоступен — всюду в блокноте вместо `!java Файл.java` используйте два шага: `!javac Файл.java && java Файл`.

Проверка «живого» запуска (ячейка 1.2 записывает файл, 1.3 — запускает):

In [12]:
%%writefile Hello.java
public class Hello {
    public static void main(String[] args) {
        System.out.println("Среда готова, Java работает: " + System.getProperty("java.version"));
    }
}

Writing Hello.java


In [13]:
!java Hello.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Среда готова, Java работает: 21.0.12.1


---

## 2. Образец: Strategy (разобран полностью)

**Сценарий.** Расчёт платы за перевозку груза. Плата зависит от вида отправки: повагонная, контейнерная, маршрутная. Формулы учебные.

### 2.1 Код «до»: один метод знает про все виды отправок

In [ ]:
%%writefile TariffBefore.java
// Код «до»: вид отправки разбирается условным оператором внутри одного метода.
public class TariffBefore {

    public static void main(String[] args) {
        System.out.printf("повагонная   : %10.2f%n", cost("WAGON", 60, 500));
        System.out.printf("контейнерная : %10.2f%n", cost("CONTAINER", 24, 500));
        System.out.printf("маршрутная   : %10.2f%n", cost("BLOCK_TRAIN", 3000, 500));
        System.out.printf("рефрижератор : %10.2f%n", cost("REFRIGERATED", 60, 500));
    }

    static double cost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;   // про новый вид отправки метод не знает
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

In [ ]:
!java TariffBefore.java

### 2.2 Разбор: что именно плохо

Запустив код «до», обратите внимание:

* рефрижераторная отправка дала **0.00** — метод о ней не знает, и единственный способ это исправить — залезть внутрь уже работающего метода;
* каждый новый вид отправки удлиняет один и тот же метод, то есть нарушает **принцип открытости/закрытости** (открыт для расширения, закрыт для изменения);
* метод одновременно отвечает за выбор варианта и за сам расчёт — нарушение **единственной ответственности**;
* протестировать один вид отправки отдельно от остальных нельзя.

**Что делает Strategy.** Каждый вариант расчёта становится отдельным классом с общим интерфейсом, а вызывающий код работает с интерфейсом и больше не знает перечня вариантов.

### 2.3 Код «после»: вариант расчёта вынесен в стратегию

In [8]:
%%writefile TariffAfter.java
// Код «после»: каждый вид отправки — отдельная стратегия расчёта.
import java.util.LinkedHashMap;
import java.util.Map;

public class TariffAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог стратегий: вызывающий код собирает его, но не содержит формул
        Map<String, TariffStrategy> tariffCatalog = new LinkedHashMap<>();
        tariffCatalog.put("WAGON", new WagonTariff());
        tariffCatalog.put("CONTAINER", new ContainerTariff());
        tariffCatalog.put("BLOCK_TRAIN", new BlockTrainTariff());

        TariffCalculator tariffCalculator = new TariffCalculator();

        System.out.println("--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---");
        for (Map.Entry<String, TariffStrategy> catalogEntry : tariffCatalog.entrySet()) {
            double calculatedCost = tariffCalculator.calculate(catalogEntry.getValue(), 60, 500);
            System.out.printf("%-12s %-28s %10.2f%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), calculatedCost);
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("повагонная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("WAGON"), 60, 500),
                           legacyCost("WAGON", 60, 500)));
        check("контейнерная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("CONTAINER"), 24, 500),
                           legacyCost("CONTAINER", 24, 500)));
        check("маршрутная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("BLOCK_TRAIN"), 3000, 500),
                           legacyCost("BLOCK_TRAIN", 3000, 500)));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---");
        TariffStrategy refrigeratedTariff = new TariffStrategy() {
            public double cost(double massTons, int distanceKm) {
                return 1.80 * massTons * distanceKm / 100.0 + 600.0;
            }
            public String title() {
                return "рефрижераторная отправка";
            }
        };
        double refrigeratedCost = tariffCalculator.calculate(refrigeratedTariff, 60, 500);
        System.out.printf("%-12s %-28s %10.2f%n", "REFRIGERATED", refrigeratedTariff.title(), refrigeratedCost);
        check("новая стратегия рассчитана, класс TariffCalculator не изменён", refrigeratedCost > 0.0);
        check("в коде «до» тот же вид отправки давал нуль", legacyCost("REFRIGERATED", 60, 500) == 0.0);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // --- служебное ---

    static boolean equalMoney(double firstValue, double secondValue) {
        return Math.abs(firstValue - secondValue) < 0.01;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Старая реализация сохранена только как эталон поведения для характеризующих тестов
    static double legacyCost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

// --- участники паттерна Strategy ---

/** Стратегия: общий интерфейс всех вариантов расчёта. */
interface TariffStrategy {
    double cost(double massTons, int distanceKm);
    String title();
}

/** Контекст: умеет считать, но не знает, какие бывают виды отправок. */
class TariffCalculator {
    double calculate(TariffStrategy tariffStrategy, double massTons, int distanceKm) {
        double calculatedValue = tariffStrategy.cost(massTons, distanceKm);
        return Math.round(calculatedValue * 100.0) / 100.0;
    }
}

class WagonTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 1.20 * massTons * distanceKm / 100.0 + 150.0;
    }
    public String title() {
        return "повагонная отправка";
    }
}

class ContainerTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.90 * massTons * distanceKm / 100.0 + 400.0;
    }
    public String title() {
        return "контейнерная отправка";
    }
}

class BlockTrainTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.70 * massTons * distanceKm / 100.0;
    }
    public String title() {
        return "маршрутная отправка";
    }
}

Writing TariffAfter.java


In [ ]:
!java TariffAfter.java

### 2.4 Диаграмма классов образца

Установка PlantUML занимает около минуты и выполняется один раз за сеанс.

In [1]:
!apt-get -qq install -y plantuml > /dev/null && echo "PlantUML установлен"

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-21/openjdk-21-jre_21.0.12.1%2b1-1%7e24.04.4_amd64.deb  404  Not Found [IP: 91.189.92.24 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [9]:
%%writefile strategy.puml
@startuml
title Strategy: расчёт платы за перевозку

interface TariffStrategy {
  +cost(massTons, distanceKm) : double
  +title() : String
}

class TariffCalculator {
  +calculate(tariffStrategy, massTons, distanceKm) : double
}

class WagonTariff
class ContainerTariff
class BlockTrainTariff

TariffCalculator o--> TariffStrategy : использует
TariffStrategy <|.. WagonTariff
TariffStrategy <|.. ContainerTariff
TariffStrategy <|.. BlockTrainTariff
@enduml

Writing strategy.puml


In [ ]:
!plantuml -tpng strategy.puml
from IPython.display import Image
Image("strategy.png")

### 2.5 Вопросы к образцу (ответить текстом в этой ячейке)

1. Какие классы пришлось бы изменить в коде «до» и в коде «после», чтобы добавить рефрижераторную отправку?
   **Ответ:** _(заполнить)_
2. Зачем в файле оставлен метод `legacyCost`, если он больше не используется в расчёте?
   **Ответ:** _(заполнить)_
3. Что произойдёт, если в каталоге не окажется запрошенного вида отправки, и как это следует обработать?
   **Ответ:** _(заполнить)_

---

## 3. Задача 1: Factory Method

**Сценарий.** Описание вагона по его типу: крытый, платформа, цистерна. Ниже — код «до», построенный на той же ошибке: условный оператор вместо полиморфизма.

### 3.1 Код «до» (запустить, не изменяя)

In [10]:
%%writefile FactoryAfter.java
// 3.3 Реализация Factory Method для закупок МТР
import java.util.ArrayList;
import java.util.List;

public class FactoryAfter {
    private static int failedChecks = 0;

    public static void main(String[] args) {
        System.out.println("--- Создание заявок через фабричный метод ---");

        // Создаем создателей (фабрики) для разных типов заявок
        RequestCreator equipmentCreator = new EquipmentRequestCreator();
        RequestCreator officeCreator = new OfficeRequestCreator();
        RequestCreator serviceCreator = new ServiceRequestCreator();

        // Создаем сами заявки
        PurchaseRequest req1 = equipmentCreator.createRequest(150000.0);
        PurchaseRequest req2 = officeCreator.createRequest(5000.0);
        PurchaseRequest req3 = serviceCreator.createRequest(30000.0);

        System.out.println(req1.description());
        System.out.println(req2.description());
        System.out.println(req3.description());

        System.out.println("\n--- Проверки ---");
        check("Заявка на оборудование содержит сумму и описание",
                req1.description().contains("150000") && !req1.description().isEmpty());
        check("Заявка на канцелярию создана правильной фабрикой",
                req2 instanceof OfficeRequest);
        check("Заявка на услуги создана правильной фабрикой",
                req3 instanceof ServiceRequest);

        // 4-й тип без правки старых классов
        System.out.println("\n--- Новый тип (Заявка на IT-услуги) ---");
        RequestCreator itCreator = new ITRequestCreator();
        PurchaseRequest req4 = itCreator.createRequest(80000.0);
        System.out.println(req4.description());
        check("Четвертый тип работает наравне с остальными",
                req4 instanceof ITRequest && req4.description().contains("80000"));

               assert failedChecks == 0 : "Не пройдено проверок: " + failedChecks;
        System.out.println("\nВсе проверки пройдены.");
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) failedChecks++;
    }
}

// --- Участники паттерна Factory Method ---

// Продукт (интерфейс)
interface PurchaseRequest {
    String description();
    double amount();
}

// Конкретные продукты
class EquipmentRequest implements PurchaseRequest {
    private final double amount;
    public EquipmentRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку оборудования на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

class OfficeRequest implements PurchaseRequest {
    private final double amount;
    public OfficeRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку канцелярии на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

class ServiceRequest implements PurchaseRequest {
    private final double amount;
    public ServiceRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку услуг на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

// 4-й продукт (IT-услуги)
class ITRequest implements PurchaseRequest {
    private final double amount;
    public ITRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на IT-услуги на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

// Создатель (фабричный метод)
interface RequestCreator {
    PurchaseRequest createRequest(double amount);
}

// Конкретные создатели
class EquipmentRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new EquipmentRequest(amount); }
}

class OfficeRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new OfficeRequest(amount); }
}

class ServiceRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new ServiceRequest(amount); }
}

// 4-й создатель
class ITRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new ITRequest(amount); }
}

Overwriting FactoryAfter.java


In [ ]:
!java FactoryBefore.java

### 3.2 Задание

Переписать сценарий на **Factory Method** в файле `FactoryAfter.java`. Требования:

1. Общий тип продукта: интерфейс или абстрактный класс `Wagon` с методами `description()` и `capacityTons()`.
2. Не менее трёх конкретных продуктов: `CoveredWagon`, `Platform`, `TankWagon`.
3. Создатель с фабричным методом: абстрактный класс или интерфейс `WagonCreator` с методом `createWagon(double capacityTons)` и конкретные создатели для каждого типа.
4. Вызывающий код (`main`) работает **только** с типами `WagonCreator` и `Wagon` и не содержит ни одного `if` по типу вагона.
5. Четвёртый тип (`HopperWagon`, зерновоз) добавляется **новым классом**, без правки ранее написанных классов; это должно быть видно по коду.
6. Проверки в файле (метод `check`, как в образце), минимум четыре:
   * описание каждого из трёх типов не пустое и содержит грузоподъёмность;
   * тип созданного объекта соответствует ожидаемому создателю;
   * четвёртый тип работает наравне с остальными;
   * метод `main` не содержит разбора типов (проверяется глазами преподавателя, в код выносить не нужно).
7. При неуспешной проверке — `throw new AssertionError(...)`.

**Подсказка по структуре файла:** первым идёт `public class FactoryAfter` с `main`, после него — интерфейсы и классы паттерна (как в `TariffAfter.java`).

In [4]:
%%writefile FactoryAfter.java
// 3.3 Реализация Factory Method для закупок МТР
import java.util.ArrayList;
import java.util.List;

public class FactoryAfter {
    private static int failedChecks = 0;

    public static void main(String[] args) {
        System.out.println("--- Создание заявок через фабричный метод ---");

        // Создаем создателей (фабрики) для разных типов заявок
        RequestCreator equipmentCreator = new EquipmentRequestCreator();
        RequestCreator officeCreator = new OfficeRequestCreator();
        RequestCreator serviceCreator = new ServiceRequestCreator();

        // Создаем сами заявки
        PurchaseRequest req1 = equipmentCreator.createRequest(150000.0);
        PurchaseRequest req2 = officeCreator.createRequest(5000.0);
        PurchaseRequest req3 = serviceCreator.createRequest(30000.0);

        System.out.println(req1.description());
        System.out.println(req2.description());
        System.out.println(req3.description());

        System.out.println("\n--- Проверки ---");
        check("Заявка на оборудование содержит сумму и описание",
                req1.description().contains("150000") && !req1.description().isEmpty());
        check("Заявка на канцелярию создана правильной фабрикой",
                req2 instanceof OfficeRequest);
        check("Заявка на услуги создана правильной фабрикой",
                req3 instanceof ServiceRequest);

        // 4-й тип без правки старых классов
        System.out.println("\n--- Новый тип (Заявка на IT-услуги) ---");
        RequestCreator itCreator = new ITRequestCreator();
        PurchaseRequest req4 = itCreator.createRequest(80000.0);
        System.out.println(req4.description());
        check("Четвертый тип работает наравне с остальными",
                req4 instanceof ITRequest && req4.description().contains("80000"));

        if (failedChecks > 0) {
            throw new AssertionError("Не пройдено проверок: " + failedChecks);
        }
        System.out.println("\nВсе проверки пройдены.");
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) failedChecks++;
    }
}

// --- Участники паттерна Factory Method ---

// Продукт (интерфейс)
interface PurchaseRequest {
    String description();
    double amount();
}

// Конкретные продукты
class EquipmentRequest implements PurchaseRequest {
    private final double amount;
    public EquipmentRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку оборудования на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

class OfficeRequest implements PurchaseRequest {
    private final double amount;
    public OfficeRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку канцелярии на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

class ServiceRequest implements PurchaseRequest {
    private final double amount;
    public ServiceRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на закупку услуг на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

// 4-й продукт (IT-услуги)
class ITRequest implements PurchaseRequest {
    private final double amount;
    public ITRequest(double amount) { this.amount = amount; }
    public String description() { return "Заявка на IT-услуги на сумму " + amount + " руб."; }
    public double amount() { return amount; }
}

// Создатель (фабричный метод)
interface RequestCreator {
    PurchaseRequest createRequest(double amount);
}

// Конкретные создатели
class EquipmentRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new EquipmentRequest(amount); }
}

class OfficeRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new OfficeRequest(amount); }
}

class ServiceRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new ServiceRequest(amount); }
}

// 4-й создатель
class ITRequestCreator implements RequestCreator {
    public PurchaseRequest createRequest(double amount) { return new ITRequest(amount); }
}

Writing FactoryAfter.java


In [5]:
!java FactoryAfter.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
--- Создание заявок через фабричный метод ---
Заявка на закупку оборудования на сумму 150000.0 руб.
Заявка на закупку канцелярии на сумму 5000.0 руб.
Заявка на закупку услуг на сумму 30000.0 руб.

--- Проверки ---
[OK]   Заявка на оборудование содержит сумму и описание
[OK]   Заявка на канцелярию создана правильной фабрикой
[OK]   Заявка на услуги создана правильной фабрикой

--- Новый тип (Заявка на IT-услуги) ---
Заявка на IT-услуги на сумму 80000.0 руб.
[OK]   Четвертый тип работает наравне с остальными

Все проверки пройдены.


In [4]:
%%writefile factory.puml
@startuml
title Factory Method: Создание заявок на закупку МТР

interface PurchaseRequest {
  +description() : String
  +amount() : double
}

class EquipmentRequest
class OfficeRequest
class ServiceRequest
class ITRequest

interface RequestCreator {
  +createRequest(amount) : PurchaseRequest
}

class EquipmentRequestCreator
class OfficeRequestCreator
class ServiceRequestCreator
class ITRequestCreator

PurchaseRequest <|.. EquipmentRequest
PurchaseRequest <|.. OfficeRequest
PurchaseRequest <|.. ServiceRequest
PurchaseRequest <|.. ITRequest

RequestCreator <|.. EquipmentRequestCreator
RequestCreator <|.. OfficeRequestCreator
RequestCreator <|.. ServiceRequestCreator
RequestCreator <|.. ITRequestCreator

EquipmentRequestCreator ..> EquipmentRequest : создает
OfficeRequestCreator ..> OfficeRequest : создает
ServiceRequestCreator ..> ServiceRequest : создает
ITRequestCreator ..> ITRequest : создает
@enduml

Writing factory.puml


In [ ]:
!plantuml -tpng factory.puml
from IPython.display import Image
Image("factory.png")

**Вывод по задаче 1** (чем создание через фабричный метод отличается от создания через `new` в вызывающем коде, и когда фабрика излишня): _(заполнить)_

---

## 4. Задача 2: Observer

**Сценарий.** Станция фиксирует прибытие поезда и должна оповестить получателей: табло, журнал, рассылку. В коде «до» станция сама знает про всех получателей.

### 4.1 Код «до» (запустить, не изменяя)

In [ ]:
%%writefile ObserverBefore.java
// Код «до»: источник события жёстко связан со всеми получателями.
public class ObserverBefore {

    public static void main(String[] args) {
        Station station = new Station("Гомель");
        station.trainArrived("3022");
        station.trainArrived("6701");
    }
}

class Station {

    private final String stationName;

    Station(String stationName) {
        this.stationName = stationName;
    }

    void trainArrived(String trainNumber) {
        // Любой новый получатель требует правки этого метода
        System.out.println("[табло]  Поезд " + trainNumber + " прибыл на станцию " + stationName);
        System.out.println("[журнал] " + stationName + ": зафиксировано прибытие поезда " + trainNumber);
    }
}

In [ ]:
!java ObserverBefore.java

### 4.2 Задание

Переписать сценарий на **Observer** в файле `ObserverAfter.java`. Требования:

1. Интерфейс наблюдателя `ArrivalListener` с методом `onTrainArrived(String stationName, String trainNumber)`.
2. Субъект `Station` с методами `addListener`, `removeListener`, `trainArrived`; класс `Station` **не знает**, какие именно наблюдатели существуют.
3. Не менее трёх наблюдателей: табло, журнал, рассылка. Журнал должен **накапливать** записи, чтобы их можно было проверить.
4. Четвёртый наблюдатель (например, передача в АСУ) добавляется без правки класса `Station`.
5. Проверки в файле, минимум четыре:
   * после одного события журнал получил ровно одну запись;
   * после отписки наблюдателя его записи больше не появляются;
   * все подписчики получили одно и то же событие;
   * добавление четвёртого наблюдателя не потребовало изменения `Station`.
6. В выводе указать порядок получения события подписчиками и ответить на вопрос в выводе ниже.

**Подсказка:** список наблюдателей удобно хранить в `java.util.ArrayList`, обход — обычным `for`.

In [5]:
%%writefile ObserverAfter.java
// 4.3 Реализация Observer для статусов заявок МТР
import java.util.ArrayList;
import java.util.List;

public class ObserverAfter {
    private static int failedChecks = 0;

    public static void main(String[] args) {
        System.out.println("--- Система оповещения о статусах заявок ---");

        PurchaseOrder order = new PurchaseOrder("Заявка №102 (Компьютеры)");

        // Создаем наблюдателей
        LogObserver logObserver = new LogObserver();
        EmailObserver emailObserver = new EmailObserver();
        WarehouseObserver warehouseObserver = new WarehouseObserver();

        // Подписываем
        order.addListener(logObserver);
        order.addListener(emailObserver);
        order.addListener(warehouseObserver);

        // Событие 1
        order.changeStatus("На согласовании");

        System.out.println("\n--- Проверки ---");
        check("Журнал получил ровно одну запись", logObserver.getLogs().size() == 1);
        check("Все подписчики получили событие",
                logObserver.getLogs().get(0).contains("На согласовании") &&
                emailObserver.lastMessage.contains("На согласовании") &&
                warehouseObserver.lastMessage.contains("На согласовании"));

        // Отписка
        System.out.println("\n--- Отписка склада ---");
        order.removeListener(warehouseObserver);
        order.changeStatus("Согласована");

        check("Отписанный наблюдатель не получил новое событие",
                warehouseObserver.lastMessage.contains("На согласовании")); // осталось старое
        check("Журнал получил вторую запись", logObserver.getLogs().size() == 2);

        // 4-й наблюдатель без правки класса PurchaseOrder
        System.out.println("\n--- Добавление 4-го наблюдателя (АСУ) ---");
        AsuObserver asuObserver = new AsuObserver();
        order.addListener(asuObserver);
        order.changeStatus("Оплачена");
        check("Новый наблюдатель работает без правки класса PurchaseOrder",
                asuObserver.lastMessage.contains("Оплачена"));

        if (failedChecks > 0) {
            throw new AssertionError("Не пройдено проверок: " + failedChecks);
        }
        System.out.println("\nВсе проверки пройдены.");
        System.out.println("\nПримечание: порядок получения события соответствует порядку подписки.");
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) failedChecks++;
    }
}

// --- Участники паттерна Observer ---

// Наблюдатель
interface StatusListener {
    void onStatusChanged(String requestName, String newStatus);
}

// Субъект
class PurchaseOrder {
    private final String requestName;
    private final List<StatusListener> listeners = new ArrayList<>();

    public PurchaseOrder(String requestName) {
        this.requestName = requestName;
    }

    public void addListener(StatusListener listener) {
        listeners.add(listener);
    }

    public void removeListener(StatusListener listener) {
        listeners.remove(listener);
    }

    public void changeStatus(String newStatus) {
        System.out.println("\n[Заявка] " + requestName + " сменила статус на: " + newStatus);
        for (StatusListener listener : listeners) {
            listener.onStatusChanged(requestName, newStatus);
        }
    }
}

// Конкретные наблюдатели
class LogObserver implements StatusListener {
    private final List<String> logs = new ArrayList<>();
    public void onStatusChanged(String requestName, String newStatus) {
        String log = "[Журнал] " + requestName + " -> " + newStatus;
        logs.add(log);
        System.out.println(log);
    }
    public List<String> getLogs() { return logs; }
}

class EmailObserver implements StatusListener {
    public String lastMessage = "";
    public void onStatusChanged(String requestName, String newStatus) {
        lastMessage = "[Email] Уведомление: " + requestName + " теперь " + newStatus;
        System.out.println(lastMessage);
    }
}

class WarehouseObserver implements StatusListener {
    public String lastMessage = "";
    public void onStatusChanged(String requestName, String newStatus) {
        lastMessage = "[Склад] Проверить наличие ТМЦ для: " + requestName + " (статус: " + newStatus + ")";
        System.out.println(lastMessage);
    }
}

// 4-й наблюдатель
class AsuObserver implements StatusListener {
    public String lastMessage = "";
    public void onStatusChanged(String requestName, String newStatus) {
        lastMessage = "[АСУ] Передача данных в систему: " + requestName + " (" + newStatus + ")";
        System.out.println(lastMessage);
    }
}

Writing ObserverAfter.java


In [ ]:
!java ObserverAfter.java

In [6]:
%%writefile observer.puml
@startuml
title Observer: Смена статуса заявки на закупку

interface StatusListener {
  +onStatusChanged(requestName, newStatus)
}

class PurchaseOrder {
  -requestName : String
  -listeners : List<StatusListener>
  +addListener(listener)
  +removeListener(listener)
  +changeStatus(newStatus)
}

class LogObserver
class EmailObserver
class WarehouseObserver
class AsuObserver

PurchaseOrder o--> "*" StatusListener : оповещает
StatusListener <|.. LogObserver
StatusListener <|.. EmailObserver
StatusListener <|.. WarehouseObserver
StatusListener <|.. AsuObserver
@enduml

Writing observer.puml


In [ ]:
!plantuml -tpng observer.puml
from IPython.display import Image
Image("observer.png")

**Вывод по задаче 2** (какую связанность снял Observer и какие новые проблемы он создал — порядок уведомлений, ошибка в одном подписчике, забытая отписка): _(заполнить)_

---

## 5. Сравнение и выводы

### 5.1 Итоговая таблица

| Паттерн | Какую проблему кода «до» устраняет | Что добавляется без правки старого кода | Чем приходится заплатить | Где применим в вашем проекте |
|---|---|---|---|---|
| Strategy |  |  |  |  |
| Factory Method |  |  |  |  |
| Observer |  |  |  |  |

### 5.2 Контрольные вопросы

1. Какой принцип SOLID нарушает разбор типа условным оператором и как каждый из трёх паттернов его восстанавливает?
   **Ответ:** Нарушается принцип Open/Closed. Strategy выносит алгоритмы в отдельные классы; Factory Method выносит создание объектов в отдельные фабрики; Observer позволяет добавлять новых подписчиков без изменения кода источника.
2. Чем Factory Method отличается от статического метода создания (`static factory method`) и от Abstract Factory?
   **Ответ:** Статический метод — это просто удобная обертка над new, не являющаяся паттерном. Abstract Factory создает целое семейство связанных объектов, а Factory Method создает один тип продукта.
3. Strategy и State имеют почти одинаковую структуру классов. В чём разница по назначению?
   **Ответ:**  клиент сам выбирает алгоритм и может менять его на лету. State — объект меняет поведение автоматически при смене внутреннего состояния.
4. Почему Observer может привести к утечке памяти и как этого избежать?
   **Ответ:** Если наблюдатель не отписался, субъект продолжает хранить на него ссылку (объект не удаляется сборщиком мусора). Решение: использовать WeakReference или обязательно вызывать removeListener.
5. Приведите пример, когда введение паттерна **вредно**. Что такое преждевременное усложнение?
   **Ответ:** Паттерн вреден, когда система простая и не будет расширяться. Преждевременное усложнение (Overengineering) — это добавление архитектурных излишеств ради "красоты", хотя бизнес-требования этого не требуют.

### 5.3 Три вывода по работе в целом

1. Паттерны проектирования — это шаблоны решения типовых архитектурных проблем.
2. Их главная ценность — снижение связанности компонентов и устойчивость к добавлению новых функций.
3. Применять паттерны нужно только тогда, когда есть реальная потребность в гибкости, иначе они усложняют код.

---

## 6. Варианты предметных областей

Номер варианта — по номеру студента в списке группы. Образец (раздел 2) выполняется всеми одинаково; задачи 1 и 2 — на своей предметной области.

| № | Предметная область | Фабрика создаёт | Стратегия варьирует | Наблюдатели получают событие |
|---|---|---|---|---|
| 1 | Грузовая станция | тип вагона | расчёт платы за пользование вагонами | вагон подан под погрузку |
| 2 | Локомотивное депо | вид ремонта (ТО-2, ТР-1, ТР-2) | приоритет заявок на ремонт | локомотив выдан из ремонта |
| 3 | Пассажирская касса | тип проездного документа | начисление скидки | рейс задержан |
| 4 | Контейнерный терминал | тип контейнера | выбор места на площадке | контейнер прибыл |
| 5 | Автотранспортное предприятие | тип транспортного средства | расчёт расхода топлива | путевой лист закрыт |
| 6 | Склад | тип тары | правило размещения (FIFO, ABC) | запас ниже нормы |
| 7 | Сортировочная станция | тип отцепа | очерёдность расформирования | состав расформирован |
| 8 | Пригородные перевозки | тип абонемента | тарификация по зонам | изменено расписание |
| 9 | Пункт таможенного оформления | тип декларации | выбор канала контроля | груз оформлен |
| 10 | Экспедиторская компания | тип заявки (авто, ж. д., море) | выбор перевозчика по критерию | статус отправки изменён |
| 11 | Путевое хозяйство | тип дефекта рельса | назначение вида работ | обнаружен дефект |
| 12 | Энергодиспетчерский пункт | тип потребителя | тарифный план (день, ночь) | авария на подстанции |
| 13 | Вагонное депо | тип неисправности | выбор отцепочного или безотцепочного ремонта | вагон забракован |
| 14 | Логистический центр | тип упаковки | правило консолидации отправок | сформирована отправка |
| 15 | Центр фирменного транспортного обслуживания | тип договора | расчёт скидки за объём | заявка согласована |

---

## 7. Чек-лист самопроверки

- [ ] Раздел 1 выполнен, версия Java не ниже 11
- [ ] `TariffBefore.java` и `TariffAfter.java` запущены, вывод сохранён
- [ ] Отвечены вопросы к образцу (п. 2.5)
- [ ] `FactoryAfter.java` написан, все проверки `[OK]`, в `main` нет разбора типов
- [ ] Четвёртый тип вагона добавлен новым классом
- [ ] `ObserverAfter.java` написан, все проверки `[OK]`, отписка работает
- [ ] Четвёртый наблюдатель добавлен без правки класса `Station`
- [ ] Построены три диаграммы: `strategy.puml`, `factory.puml`, `observer.puml`
- [ ] Заполнена итоговая таблица (п. 5.1)
- [ ] Отвечены пять контрольных вопросов и сформулированы три вывода
- [ ] Блокнот выполняется сверху вниз без ошибок после *Среда выполнения → Перезапустить и выполнить все*
- [ ] Запущена ячейка самопроверки, её вывод вставлен в описание pull request

## 8. Ячейка самопроверки

Ячейка сама компилирует и запускает все три файла, проверяет отсутствие `[FAIL]`, наличие диаграмм и минимальные признаки соблюдения требований. Запускать после того, как задачи выполнены.

In [11]:
#@title Запустить самопроверку и получить отчёт
import os, re, subprocess, datetime

ФИО_исполнителя = "Редько Т.В."  #@param {type:"string"}
Группа_исполнителя = "СИ-41"  #@param {type:"string"}
Номер_варианта = 17  #@param {type:"integer"}
ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ПРОГРАММЫ = [
    ("Образец, Strategy", "TariffAfter.java"),
    ("Задача 1, Factory Method", "FactoryAfter.java"),
    ("Задача 2, Observer", "ObserverAfter.java"),
]
ДИАГРАММЫ = ["strategy.puml", "factory.puml", "observer.puml"]

def запустить(имя_файла):
    """Запустить один исходный файл Java и вернуть (успех, пояснение)."""
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    if завершение.returncode != 0:
        значимые = [s for s in вывод.splitlines() if s.strip() and not s.startswith("Picked up")]
        первая_строка = значимые[0] if значимые else "без сообщения"
        return False, "аварийное завершение: " + первая_строка[:90]
    if "[FAIL]" in вывод:
        return False, "проверок не пройдено: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 3:
        return False, "проверок в файле только %d, требуется не менее 3" % успешных
    return True, "проверок пройдено: %d" % успешных

результаты = []
for название, файл in ПРОГРАММЫ:
    результаты.append((название, файл) + запустить(файл))

# Признаки соблюдения требований, которые можно проверить по тексту
замечания = []
def тело_метода(исходный_текст, сигнатура="main("):
    """Вернуть тело метода по сопоставлению фигурных скобок (без построчных комментариев)."""
    без_комментариев = re.sub(r"//[^\n]*", "", исходный_текст)
    начало = без_комментариев.find(сигнатура)
    if начало < 0:
        return ""
    открывающая = без_комментариев.find("{", начало)
    if открывающая < 0:
        return ""
    глубина = 0
    for позиция in range(открывающая, len(без_комментариев)):
        символ = без_комментариев[позиция]
        if символ == "{":
            глубина += 1
        elif символ == "}":
            глубина -= 1
            if глубина == 0:
                return без_комментариев[открывающая:позиция + 1]
    return без_комментариев[открывающая:]

if os.path.exists("FactoryAfter.java"):
    текст_фабрики = open("FactoryAfter.java", encoding="utf-8").read()
    блок_main = тело_метода(текст_фабрики)
    if re.search(r'\b(if|switch)\b', блок_main):
        замечания.append("в FactoryAfter.java метод main содержит ветвление по типу")
    if текст_фабрики.count("class ") < 5:
        замечания.append("в FactoryAfter.java меньше пяти классов: продукты, создатели и четвёртый тип, вероятно, не описаны")
if os.path.exists("ObserverAfter.java"):
    текст_наблюдателя = open("ObserverAfter.java", encoding="utf-8").read()
    for требуемое in ("addListener", "removeListener"):
        if требуемое not in текст_наблюдателя:
            замечания.append("в ObserverAfter.java нет метода %s" % требуемое)
отсутствующие_диаграммы = [d for d in ДИАГРАММЫ if not os.path.exists(d)]

строки = [
    "### Отчёт самопроверки «Песочница паттернов GoF»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s, вариант %d |" % (
        ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА", Номер_варианта),
    "| Дата и время отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "| Версия Java | %s |" % next(
        (s for s in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
         if "version" in s), "не определена"),
    "",
    "| Программа | Файл | Результат |",
    "|---|---|---|",
]
for название, файл, успех, пояснение in результаты:
    строки.append("| %s | `%s` | %s — %s |" % (название, файл, "пройдено" if успех else "НЕ ПРОЙДЕНО", пояснение))
строки += [
    "",
    "| Диаграммы | %s |" % ("все три на месте" if not отсутствующие_диаграммы
                            else "отсутствуют: " + ", ".join(отсутствующие_диаграммы)),
    "| Таблица и выводы | %s |" % ("заполнены" if ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
    "| Замечания | %s |" % ("нет" if not замечания else "; ".join(замечания)),
]
print("\n".join(строки))
print("\nСкопируйте отчёт выше в описание pull request.\n")

ошибки = []
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
ошибки += ["%s: %s" % (название, пояснение) for название, файл, успех, пояснение in результаты if not успех]
if отсутствующие_диаграммы:
    ошибки.append("нет диаграмм: " + ", ".join(отсутствующие_диаграммы))
if not ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены итоговая таблица и выводы")
ошибки += замечания

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Песочница паттернов GoF»

| Параметр | Значение |
|---|---|
| Исполнитель | Редько Т.В., группа СИ-41, вариант 17 |
| Дата и время отчёта (UTC) | 2026-11-17 09:09 |
| Версия Java | openjdk version "21.0.12.1" 2026-08-18 |

| Программа | Файл | Результат |
|---|---|---|
| Образец, Strategy | `TariffAfter.java` | пройдено — проверок пройдено: 5 |
| Задача 1, Factory Method | `FactoryAfter.java` | пройдено — проверок пройдено: 4 |
| Задача 2, Observer | `ObserverAfter.java` | пройдено — проверок пройдено: 5 |

| Диаграммы | все три на месте |
| Таблица и выводы | заполнены |
| Замечания | нет |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.


---

## 9. Критерии оценивания

| Критерий | Баллов |
|---|---|
| Образец запущен, вопросы п. 2.5 отвечены содержательно | 10 |
| Factory Method: корректная структура паттерна, в `main` нет разбора типов | 20 |
| Factory Method: четвёртый тип добавлен новым классом | 10 |
| Observer: корректная структура, работают подписка и отписка | 20 |
| Observer: четвёртый наблюдатель добавлен без правки субъекта | 10 |
| Проверки в каждом файле осмысленны (не `[OK]` ради `[OK]`) | 10 |
| Три диаграммы классов соответствуют коду | 10 |
| Итоговая таблица, контрольные вопросы, выводы | 10 |
| **Итого** | **100** |

Снижение: задача решена не на своём варианте — минус 20; код «после» повторяет разбор типа условным оператором внутри нового класса — минус 20; диаграмма не соответствует коду — минус 10.

## 10. Для желающих (сверх задания)

| Что сделать | Зачем |
|---|---|
| Подключить настоящий JUnit 5 вместо метода `check`: `!wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.10.2/junit-platform-console-standalone-1.10.2.jar` и запуск через `java -jar` | переход к инструменту, который используется в ЛР по модульному тестированию |
| Заменить три `if` в выборе стратегии на `java.util.Map` с ключом-перечислением (`enum`) | показать, что каталог стратегий сам может быть устранён из вызывающего кода |
| Реализовать Observer на `java.util.function.Consumer` вместо своего интерфейса | сравнить классический GoF и функциональный стиль |